# CRUD

In [11]:
import sys
sys.path.append("..")

import pandas as pd
from IPython.display import display
from sql.queries import QUERIES
from utils import db_connection as db

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)


def show(key, rows=15):
    """Print the question title and SQL, then show the result table."""
    q = QUERIES[key]
    print(f"{key.upper()} [{q['level']}] {q['title']}")
    print("-" * 70)
    print(q["sql"], "\n")
    df = db.run_query(q["sql"])
    print(f"{len(df)} rows")
    display(df.head(rows))
    return df

# Cell 1: setup

In [1]:
import sys
sys.path.append("..")

from services import crud_service as crud
from utils import db_connection as db

db.table_counts()

{'batting_performances': 11012,
 'bowling_performances': 5350,
 'matches': 548,
 'player_career_stats': 277,
 'players': 120,
 'series': 40,
 'teams': 115,
 'venues': 67}

# Cell 2: Create

In [2]:
good = {"full_name": "Demo Player", "country": "India",
        "playing_role": "All-rounder", "batting_style": "Right-hand bat",
        "bowling_style": "Legbreak"}
stats = {"format": "ODI", "matches": 25, "innings": 22, "runs": 700,
         "highest_score": 98, "centuries": 0, "fifties": 5, "wickets": 30}

result = crud.create_player(good, stats)
print(result)
player_id = result.player_id

Result(ok=True, message='Added Demo Player (ID 1000000).', player_id=1000000, errors=[])


# Cell 3: Read

In [3]:
print(crud.get_player(player_id))
crud.get_stats(player_id)

{'player_id': 1000000, 'full_name': 'Demo Player', 'country': 'India', 'playing_role': 'All-rounder', 'batting_style': 'Right-hand bat', 'bowling_style': 'Legbreak'}


,player_id,format,matches,innings,runs,highest_score,batting_avg,strike_rate,centuries,fifties,wickets,bowling_avg,economy_rate,catches,stumpings
0,1000000,ODI,25,22,700,98,None,None,0,5,30,None,None,0,0


In [4]:
crud.list_players(search="demo")

,player_id,full_name,country,playing_role,batting_style,bowling_style
0,1000000,Demo Player,India,All-rounder,Right-hand bat,Legbreak


# Cell 4: Update

In [5]:
print(crud.update_player(player_id, dict(good, country="Australia")))
print(crud.save_stats(player_id, dict(stats, format="ODI", runs=850, highest_score=110)))
crud.get_player(player_id)

Result(ok=True, message='Updated Demo Player.', player_id=1000000, errors=[])
Result(ok=True, message='Saved ODI stats.', player_id=1000000, errors=[])


{'player_id': 1000000,
 'full_name': 'Demo Player',
 'country': 'Australia',
 'playing_role': 'All-rounder',
 'batting_style': 'Right-hand bat',
 'bowling_style': 'Legbreak'}

# Cell 5: Validation blocks bad input

In [6]:
bad_inputs = [
    ("name too short",    dict(good, full_name="X")),
    ("digits in name",    dict(good, full_name="Player 99")),
    ("unknown role",      dict(good, playing_role="Umpire")),
    ("bowler, no style",  dict(good, full_name="Other Bowler", playing_role="Bowler", bowling_style=None)),
    ("duplicate",         good),
]
for label, data in bad_inputs:
    r = crud.create_player(data)
    print(f"{'saved (bad!)' if r.ok else 'blocked'}: {label:18} -> {r.message}")

blocked: name too short     -> Name must be 2-60 characters: letters, spaces, . ' - only.
blocked: digits in name     -> Name must be 2-60 characters: letters, spaces, . ' - only.
blocked: unknown role       -> Playing role must be one of: Batsman, Bowler, All-rounder, Wicket-keeper.
blocked: bowler, no style   -> Bowlers and all-rounders need a bowling style.
saved (bad!): duplicate          -> Added Demo Player (ID 1000001).


# Cell 6: Bad stats roll the whole create back

In [7]:
before = db.table_counts()["players"]
r = crud.create_player(dict(good, full_name="Rollback Test"),
                       dict(stats, highest_score=9999))
print(r.message)
print("players before/after:", before, db.table_counts()["players"])

Highest Score must be between 0 and 500.
players before/after: 122 122


# Cell 7: SQL injection attempt

In [8]:
evil = "'; DROP TABLE players; --"
print("search result rows:", len(crud.list_players(search=evil)))
print("create attempt ok? ", crud.create_player(dict(good, full_name=evil)).ok)
print("players table still exists:", "players" in db.table_counts())

search result rows: 0
create attempt ok?  False
players table still exists: True


# Cell 8: Delete

In [9]:
print(crud.player_footprint(player_id))
print(crud.delete_player(player_id))
print(crud.get_player(player_id))   # None

{'career_stats': 1, 'batting_rows': 0, 'bowling_rows': 0}
Result(ok=True, message='Deleted Demo Player and 1 related rows.', player_id=1000000, errors=[])
None


# Cell 9: Delete a seed player who has match data

In [10]:
busiest = db.run_query(
    "SELECT player_id FROM batting_performances GROUP BY 1 "
    "ORDER BY COUNT(*) DESC LIMIT 1")["player_id"].iloc[0]
busiest = int(busiest)
print(crud.player_footprint(busiest))
print(crud.delete_player(busiest))
print("foreign-key problems:", len(db.run_query("PRAGMA foreign_key_check")))

{'career_stats': 3, 'batting_rows': 141, 'bowling_rows': 0}
Result(ok=True, message='Deleted Hamish Pickett and 144 related rows.', player_id=900061, errors=[])
foreign-key problems: 0
